# Artist Popularity Enrichment

Build a resumable artist popularity lookup from the processed training artists. The workflow normalizes artist names, resolves MusicBrainz artist IDs, queries Last.fm `artist.getInfo`, builds an `artist_popularity` dictionary, and saves the lookup to parquet.

## Setup

Set `MAX_ARTISTS` to a small value for smoke tests. Set it to `None` or export `MAX_ARTISTS=None` for the full run. MusicBrainz is limited to about one request per second, so the full artist list can take several hours.

In [9]:
from __future__ import annotations

import os
import re
import time
import unicodedata
from dataclasses import dataclass
from datetime import datetime, timezone
from difflib import SequenceMatcher
from pathlib import Path
from typing import Any

import pandas as pd
import requests

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

INPUT_PARQUET = PROJECT_ROOT / "data" / "processed" / "orig_data.parquet"
OUTPUT_PARQUET = PROJECT_ROOT / "data" / "interim" / "artist_popularity.parquet"

# Override with environment variables when running headless, for example:
# MAX_ARTISTS=3 python -m jupyter nbconvert --execute --to notebook --inplace notebooks/test.ipynb
LASTFM_API_KEY = os.getenv("LASTFM_API_KEY", "d65449d5841d9c51caef29fa2398aa03").strip()
MUSICBRAINZ_USER_AGENT = os.getenv(
    "MUSICBRAINZ_USER_AGENT",
    "SongAssess/0.1 (https://github.com/sonomilano/SongAssess)",
).strip()
FORCE_REFRESH = os.getenv("FORCE_REFRESH", "false").strip().lower() in {"1", "true", "yes", "y"}
SAVE_EVERY = int(os.getenv("SAVE_EVERY", "10"))


def parse_optional_int(value: str | None, default: int | None) -> int | None:
    if value is None or value == "":
        return default
    if value.strip().lower() in {"none", "null", "all"}:
        return None
    return int(value)


MAX_ARTISTS = parse_optional_int(os.getenv("MAX_ARTISTS"), None)

MUSICBRAINZ_ARTIST_SEARCH_URL = "https://musicbrainz.org/ws/2/artist/"
LASTFM_API_URL = "https://ws.audioscrobbler.com/2.0/"
REQUEST_TIMEOUT_SECONDS = 20
MAX_REQUEST_ATTEMPTS = 4
TRANSIENT_STATUS_CODES = {429, 500, 502, 503, 504}
COMPLETED_STATUSES = {"ok", "ok_name_fallback", "ok_without_musicbrainz"}
MUSICBRAINZ_MIN_NAME_SIMILARITY = float(os.getenv("MUSICBRAINZ_MIN_NAME_SIMILARITY", "0.80"))

OUTPUT_COLUMNS = [
    "normalized_artist_name",
    "artist_name",
    "musicbrainz_artist_id",
    "musicbrainz_name",
    "musicbrainz_score",
    "lastfm_name",
    "lastfm_url",
    "lastfm_listeners",
    "lastfm_playcount",
    "lookup_status",
    "error",
    "fetched_at_utc",
]

print(f"Input: {INPUT_PARQUET}")
print(f"Output: {OUTPUT_PARQUET}")
print(f"MAX_ARTISTS: {MAX_ARTISTS}")
print(f"FORCE_REFRESH: {FORCE_REFRESH}")

Input: /Users/macbook/ProgrammingProjects/SongAssess/popularity-model/data/processed/orig_data.parquet
Output: /Users/macbook/ProgrammingProjects/SongAssess/popularity-model/data/interim/artist_popularity.parquet
MAX_ARTISTS: None
FORCE_REFRESH: False


## Helpers

In [10]:
def normalize_artist_name(value: Any) -> str:
    """Normalize an artist name for matching and dictionary keys."""
    if pd.isna(value):
        return ""
    text = unicodedata.normalize("NFKC", str(value))
    text = text.casefold().strip()
    return re.sub(r"\s+", " ", text)


@dataclass
class RateLimiter:
    min_interval_seconds: float
    last_call_monotonic: float = 0.0

    def wait(self) -> None:
        elapsed = time.monotonic() - self.last_call_monotonic
        remaining = self.min_interval_seconds - elapsed
        if remaining > 0:
            time.sleep(remaining)
        self.last_call_monotonic = time.monotonic()


def parse_int(value: Any) -> int | None:
    if value is None or value == "":
        return None
    try:
        return int(str(value).replace(",", ""))
    except (TypeError, ValueError):
        return None


def append_error(existing: str | None, new_error: Exception | str | None) -> str | None:
    if not new_error:
        return None if pd.isna(existing) else existing
    message = str(new_error)
    if existing is None or pd.isna(existing) or existing == "":
        return message
    return f"{existing}; {message}"


def request_json(
    session: requests.Session,
    url: str,
    *,
    params: dict[str, Any],
    headers: dict[str, str] | None = None,
    limiter: RateLimiter | None = None,
    max_attempts: int = MAX_REQUEST_ATTEMPTS,
) -> dict[str, Any]:
    last_error: Exception | None = None
    for attempt in range(1, max_attempts + 1):
        if limiter is not None:
            limiter.wait()

        try:
            response = session.get(
                url,
                params=params,
                headers=headers,
                timeout=REQUEST_TIMEOUT_SECONDS,
            )
            if response.status_code in TRANSIENT_STATUS_CODES:
                raise requests.HTTPError(
                    f"HTTP {response.status_code}: {response.text[:300]}",
                    response=response,
                )
            response.raise_for_status()
            return response.json()
        except (requests.RequestException, ValueError) as exc:
            last_error = exc
            if attempt == max_attempts:
                break
            retry_after = None
            response = getattr(exc, "response", None)
            if response is not None:
                retry_after = response.headers.get("Retry-After")
            if retry_after:
                try:
                    sleep_seconds = float(retry_after)
                except ValueError:
                    sleep_seconds = 2 ** (attempt - 1)
            else:
                sleep_seconds = min(30, 2 ** (attempt - 1))
            time.sleep(sleep_seconds)

    raise RuntimeError(f"Request failed after {max_attempts} attempts: {last_error}")


musicbrainz_limiter = RateLimiter(1.05)
lastfm_limiter = RateLimiter(0.21)

## Load Artists

In [11]:
if not INPUT_PARQUET.exists():
    raise FileNotFoundError(f"Input parquet not found: {INPUT_PARQUET}")

source_df = pd.read_parquet(INPUT_PARQUET, columns=["primary_artist"])
artist_table = (
    source_df["primary_artist"]
    .dropna()
    .astype(str)
    .str.strip()
    .loc[lambda series: series.ne("")]
    .drop_duplicates()
    .to_frame(name="artist_name")
)
artist_table["normalized_artist_name"] = artist_table["artist_name"].map(normalize_artist_name)
artist_table = (
    artist_table.loc[artist_table["normalized_artist_name"].ne("")]
    .drop_duplicates("normalized_artist_name", keep="first")
    .reset_index(drop=True)
)

print(f"Source rows: {len(source_df):,}")
print(f"Unique normalized artists: {len(artist_table):,}")
artist_table.head()

Source rows: 66,808
Unique normalized artists: 16,733


,artist_name,normalized_artist_name
0,Sam Smith,sam smith
1,Bizarrap,bizarrap
2,Manuel Turizo,manuel turizo
3,David Guetta,david guetta
4,Bad Bunny,bad bunny


## Load Existing Results

In [12]:
def empty_results_frame() -> pd.DataFrame:
    return pd.DataFrame(columns=OUTPUT_COLUMNS)


def normalize_output_frame(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    for column in OUTPUT_COLUMNS:
        if column not in frame.columns:
            frame[column] = pd.NA
    frame = frame[OUTPUT_COLUMNS]
    for column in ["musicbrainz_score", "lastfm_listeners", "lastfm_playcount"]:
        frame[column] = pd.to_numeric(frame[column], errors="coerce").astype("Int64")
    return frame


def load_existing_results(path: Path) -> pd.DataFrame:
    if not path.exists() or FORCE_REFRESH:
        return empty_results_frame()
    return normalize_output_frame(pd.read_parquet(path))


existing_results = load_existing_results(OUTPUT_PARQUET)
completed_names = set(
    existing_results.loc[
        existing_results["lookup_status"].isin(COMPLETED_STATUSES),
        "normalized_artist_name",
    ].dropna()
)

pending_artists = artist_table.copy()
if MAX_ARTISTS is not None:
    pending_artists = pending_artists.head(MAX_ARTISTS)
if not FORCE_REFRESH:
    pending_artists = pending_artists.loc[
        ~pending_artists["normalized_artist_name"].isin(completed_names)
    ]

print(f"Existing result rows: {len(existing_results):,}")
print(f"Completed artists skipped: {len(completed_names):,}")
print(f"Pending artists this run: {len(pending_artists):,}")
pending_artists.head()

Existing result rows: 25
Completed artists skipped: 25
Pending artists this run: 16,708


,artist_name,normalized_artist_name
25,Stephen Sanchez,stephen sanchez
26,Doja Cat,doja cat
27,Lil Nas X,lil nas x
28,Beach Weather,beach weather
29,Shakira,shakira


## API Lookup Functions

In [13]:
def musicbrainz_name_similarity(candidate: dict[str, Any], normalized_artist_name: str) -> float:
    candidate_names = [
        normalize_artist_name(candidate.get("name")),
        normalize_artist_name(candidate.get("sort-name")),
    ]
    candidate_names = [name for name in candidate_names if name]
    if not candidate_names:
        return 0.0
    return max(
        SequenceMatcher(None, normalized_artist_name, candidate_name).ratio()
        for candidate_name in candidate_names
    )


def choose_musicbrainz_artist(candidates: list[dict[str, Any]], normalized_artist_name: str) -> dict[str, Any] | None:
    if not candidates:
        return None

    def score(candidate: dict[str, Any]) -> int:
        return parse_int(candidate.get("score")) or 0

    exact_matches = [
        candidate
        for candidate in candidates
        if normalize_artist_name(candidate.get("name")) == normalized_artist_name
        or normalize_artist_name(candidate.get("sort-name")) == normalized_artist_name
    ]
    if exact_matches:
        return sorted(exact_matches, key=score, reverse=True)[0]

    plausible_matches = [
        candidate
        for candidate in candidates
        if musicbrainz_name_similarity(candidate, normalized_artist_name) >= MUSICBRAINZ_MIN_NAME_SIMILARITY
    ]
    if not plausible_matches:
        return None
    return sorted(plausible_matches, key=score, reverse=True)[0]


def search_musicbrainz_artist(
    session: requests.Session,
    artist_name: str,
    normalized_artist_name: str,
) -> dict[str, Any] | None:
    data = request_json(
        session,
        MUSICBRAINZ_ARTIST_SEARCH_URL,
        params={
            "query": artist_name,
            "fmt": "json",
            "limit": 5,
            "dismax": "true",
        },
        headers={"User-Agent": MUSICBRAINZ_USER_AGENT},
        limiter=musicbrainz_limiter,
    )
    return choose_musicbrainz_artist(data.get("artists", []), normalized_artist_name)


def request_lastfm_artist(
    session: requests.Session,
    *,
    artist_name: str | None = None,
    musicbrainz_artist_id: str | None = None,
) -> dict[str, Any]:
    if not LASTFM_API_KEY:
        raise ValueError("LASTFM_API_KEY is required for Last.fm artist.getInfo requests")

    params: dict[str, Any] = {
        "method": "artist.getInfo",
        "api_key": LASTFM_API_KEY,
        "format": "json",
    }
    if musicbrainz_artist_id:
        params["mbid"] = musicbrainz_artist_id
    elif artist_name:
        params["artist"] = artist_name
        params["autocorrect"] = 1
    else:
        raise ValueError("Either artist_name or musicbrainz_artist_id is required")

    data = request_json(
        session,
        LASTFM_API_URL,
        params=params,
        limiter=lastfm_limiter,
    )
    if "error" in data:
        raise RuntimeError(f"Last.fm error {data.get('error')}: {data.get('message')}")
    artist = data.get("artist")
    if not artist:
        raise RuntimeError("Last.fm response did not include an artist object")
    return artist


def fetch_lastfm_artist_info(
    session: requests.Session,
    artist_name: str,
    musicbrainz_artist_id: str | None,
) -> tuple[dict[str, Any], str, str | None]:
    mbid_error = None
    if musicbrainz_artist_id:
        try:
            return request_lastfm_artist(
                session,
                musicbrainz_artist_id=musicbrainz_artist_id,
            ), "mbid", None
        except Exception as exc:
            mbid_error = str(exc)

    artist = request_lastfm_artist(session, artist_name=artist_name)
    return artist, "name", mbid_error


def build_base_result(artist_name: str, normalized_artist_name: str) -> dict[str, Any]:
    return {
        "normalized_artist_name": normalized_artist_name,
        "artist_name": artist_name,
        "musicbrainz_artist_id": pd.NA,
        "musicbrainz_name": pd.NA,
        "musicbrainz_score": pd.NA,
        "lastfm_name": pd.NA,
        "lastfm_url": pd.NA,
        "lastfm_listeners": pd.NA,
        "lastfm_playcount": pd.NA,
        "lookup_status": "pending",
        "error": pd.NA,
        "fetched_at_utc": datetime.now(timezone.utc).isoformat(),
    }


def enrich_artist(session: requests.Session, artist_name: str, normalized_artist_name: str) -> dict[str, Any]:
    result = build_base_result(artist_name, normalized_artist_name)
    musicbrainz_artist_id = None
    musicbrainz_found = False

    try:
        musicbrainz_artist = search_musicbrainz_artist(session, artist_name, normalized_artist_name)
        if musicbrainz_artist:
            musicbrainz_found = True
            musicbrainz_artist_id = musicbrainz_artist.get("id")
            result["musicbrainz_artist_id"] = musicbrainz_artist_id
            result["musicbrainz_name"] = musicbrainz_artist.get("name")
            result["musicbrainz_score"] = parse_int(musicbrainz_artist.get("score"))
    except Exception as exc:
        result["error"] = append_error(result.get("error"), f"MusicBrainz: {exc}")

    try:
        lastfm_artist, lastfm_source, mbid_error = fetch_lastfm_artist_info(
            session,
            artist_name,
            musicbrainz_artist_id,
        )
        if mbid_error:
            result["error"] = append_error(result.get("error"), f"Last.fm MBID fallback: {mbid_error}")

        stats = lastfm_artist.get("stats", {}) or {}
        result["lastfm_name"] = lastfm_artist.get("name")
        result["lastfm_url"] = lastfm_artist.get("url")
        result["lastfm_listeners"] = parse_int(stats.get("listeners"))
        result["lastfm_playcount"] = parse_int(stats.get("playcount", stats.get("plays")))

        if musicbrainz_found and lastfm_source == "mbid":
            result["lookup_status"] = "ok"
        elif musicbrainz_found:
            result["lookup_status"] = "ok_name_fallback"
        else:
            result["lookup_status"] = "ok_without_musicbrainz"
    except Exception as exc:
        result["lookup_status"] = "failed"
        result["error"] = append_error(result.get("error"), f"Last.fm: {exc}")

    result["fetched_at_utc"] = datetime.now(timezone.utc).isoformat()
    return result

## Run Enrichment

In [14]:
def write_results(results_by_name: dict[str, dict[str, Any]], path: Path) -> pd.DataFrame:
    path.parent.mkdir(parents=True, exist_ok=True)
    if results_by_name:
        result_frame = pd.DataFrame(results_by_name.values())
    else:
        result_frame = empty_results_frame()
    result_frame = normalize_output_frame(result_frame)
    result_frame = result_frame.sort_values("normalized_artist_name").reset_index(drop=True)
    result_frame.to_parquet(path, index=False)
    return result_frame


results_by_name = {
    row["normalized_artist_name"]: row
    for row in normalize_output_frame(existing_results).to_dict("records")
}

session = requests.Session()
processed_count = 0

for row in pending_artists.itertuples(index=False):
    result = enrich_artist(
        session,
        artist_name=row.artist_name,
        normalized_artist_name=row.normalized_artist_name,
    )
    results_by_name[row.normalized_artist_name] = result
    processed_count += 1

    if processed_count % SAVE_EVERY == 0:
        current_results = write_results(results_by_name, OUTPUT_PARQUET)
        print(f"Saved {len(current_results):,} rows after {processed_count:,} new lookups")

artist_popularity_df = write_results(results_by_name, OUTPUT_PARQUET)
print(f"Processed new lookups: {processed_count:,}")
print(f"Saved rows: {len(artist_popularity_df):,}")
print(f"Output parquet: {OUTPUT_PARQUET}")
artist_popularity_df["lookup_status"].value_counts(dropna=False)

Saved 35 rows after 10 new lookups
Saved 45 rows after 20 new lookups
Saved 55 rows after 30 new lookups
Saved 65 rows after 40 new lookups
Saved 75 rows after 50 new lookups
Saved 85 rows after 60 new lookups
Saved 95 rows after 70 new lookups
Saved 105 rows after 80 new lookups
Saved 115 rows after 90 new lookups
Saved 125 rows after 100 new lookups
Saved 135 rows after 110 new lookups
Saved 145 rows after 120 new lookups
Saved 155 rows after 130 new lookups
Saved 165 rows after 140 new lookups
Saved 175 rows after 150 new lookups
Saved 185 rows after 160 new lookups
Saved 195 rows after 170 new lookups
Saved 205 rows after 180 new lookups
Saved 215 rows after 190 new lookups
Saved 225 rows after 200 new lookups
Saved 235 rows after 210 new lookups
Saved 245 rows after 220 new lookups
Saved 255 rows after 230 new lookups
Saved 265 rows after 240 new lookups
Saved 275 rows after 250 new lookups
Saved 285 rows after 260 new lookups
Saved 295 rows after 270 new lookups
Saved 305 rows af

lookup_status
ok                        13387
ok_without_musicbrainz     1902
ok_name_fallback           1441
failed                        3
Name: count, dtype: int64

## Build Dictionary And Check Output

In [15]:
artist_popularity = (
    artist_popularity_df
    .set_index("normalized_artist_name")
    .drop(columns=[])
    .to_dict(orient="index")
)

expected_columns = set(OUTPUT_COLUMNS)
missing_columns = expected_columns.difference(artist_popularity_df.columns)
if missing_columns:
    raise AssertionError(f"Missing output columns: {sorted(missing_columns)}")

if artist_popularity_df["normalized_artist_name"].duplicated().any():
    raise AssertionError("Output contains duplicate normalized_artist_name values")

print(f"artist_popularity dictionary entries: {len(artist_popularity):,}")
artist_popularity_df.head(10)

artist_popularity dictionary entries: 16,733


,normalized_artist_name,artist_name,musicbrainz_artist_id,musicbrainz_name,musicbrainz_score,lastfm_name,lastfm_url,lastfm_listeners,lastfm_playcount,lookup_status,error,fetched_at_utc
0,!nvite,!nvite,NaN,NaN,<NA>,!nvite,https://www.last.fm/music/%21nvite,9631,24838,ok_without_musicbrainz,NaN,2026-07-12T08:37:16.795566+00:00
1,"""weird al"" yankovic","""Weird Al"" Yankovic",7746d775-9550-4360-b8d5-c37bd448ce01,“Weird Al” Yankovic,100,“Weird Al” Yankovic,https://www.last.fm/music/%E2%80%9CWeird+Al%E2...,5341,177686,ok,NaN,2026-07-12T06:43:29.708600+00:00
2,#kids,#Kids,0bd39aa1-9223-4b28-8ac3-588153d872fb,KIDS.,85,#Kids,https://www.last.fm/music/%23Kids,886,5065,ok_name_fallback,Last.fm MBID fallback: Last.fm error 6: The ar...,2026-07-12T09:19:14.246794+00:00
3,$affie,$affie,NaN,NaN,<NA>,$affie,https://www.last.fm/music/$affie,13101,35210,ok_without_musicbrainz,NaN,2026-07-12T08:26:37.403655+00:00
4,&me,&ME,ab886374-8565-4407-ac50-f76baadd4c7f,&ME,100,&ME,https://www.last.fm/music/&ME,188541,1867406,ok,NaN,2026-07-12T07:45:44.631749+00:00
5,'falsettos' 2016 broadway company,'Falsettos' 2016 Broadway Company,NaN,NaN,<NA>,'Falsettos' 2016 Broadway Company,https://www.last.fm/music/%27Falsettos%27+2016...,31098,761122,ok_without_musicbrainz,NaN,2026-07-12T09:39:57.043766+00:00
6,'til tuesday,'Til Tuesday,8cf8869d-e066-4c94-b734-fe05749badf0,’Til Tuesday,100,’Til Tuesday,https://www.last.fm/music/%E2%80%99Til+Tuesday,1228,8628,ok,NaN,2026-07-12T09:51:25.329564+00:00
7,(dolch),(DOLCH),e89e7445-6efd-4299-8ae3-6efec30c6686,(DOLCH),100,(DOLCH),https://www.last.fm/music/(DOLCH),40516,484289,ok,NaN,2026-07-12T10:25:02.533563+00:00
8,(g)i-dle,(G)I-DLE,NaN,NaN,<NA>,(G)I-DLE,https://www.last.fm/music/(G)I-DLE,955812,104225399,ok_without_musicbrainz,NaN,2026-07-12T05:18:51.293989+00:00
9,(hed) p.e.,(Hed) P.E.,19516266-e5d9-4774-b749-812bb76a6559,(həd) p.e.,100,(həd) p.e.,https://www.last.fm/music/(h%C9%99d)+p.e.,16024,333937,ok,NaN,2026-07-12T09:51:48.673394+00:00


In [27]:
artist_data = pd.read_parquet(PROJECT_ROOT / "data" / "interim" / "artist_popularity.parquet")
artist_data.dropna(subset=['lastfm_listeners'], inplace=True)

In [28]:
artist_data.describe()

,musicbrainz_score,lastfm_listeners,lastfm_playcount
count,14828.0,16730.0,16730.0
mean,99.725317,261011.232516,9748554.14202
std,2.604739,637305.544451,66511583.352609
min,38.0,1.0,1.0
25%,100.0,11094.25,103076.0
50%,100.0,49957.0,624276.0
75%,100.0,198446.5,3321584.25
max,100.0,9220581.0,4567120470.0


In [33]:
# Add Last.fm listener counts to each track row by primary artist.
orig_data = pd.read_parquet(INPUT_PARQUET).copy()
artist_data = pd.read_parquet(OUTPUT_PARQUET).copy()

artist_data["lastfm_listeners"] = pd.to_numeric(
    artist_data["lastfm_listeners"],
    errors="coerce",
).astype("Int64")

listener_lookup = (
    artist_data
    .dropna(subset=["normalized_artist_name"])
    .drop_duplicates("normalized_artist_name", keep="last")
    .set_index("normalized_artist_name")["lastfm_listeners"]
)

orig_data["normalized_primary_artist"] = orig_data["primary_artist"].map(normalize_artist_name)
orig_data["artists_listeners"] = orig_data["normalized_primary_artist"].map(listener_lookup).astype("Int64")
orig_data = orig_data.drop(columns=["normalized_primary_artist"])

missing_listener_count = orig_data["artists_listeners"].isna().sum()
print(f"Added artists_listeners to {len(orig_data):,} rows")
print(f"Rows without matched listener count: {missing_listener_count:,}")

orig_data[["primary_artist", "artists_listeners"]].head(10)

Added artists_listeners to 66,808 rows
Rows without matched listener count: 4


,primary_artist,artists_listeners
0,Sam Smith,3158290
1,Bizarrap,954116
2,Manuel Turizo,536548
3,David Guetta,5340781
4,Bad Bunny,2679395
5,Bad Bunny,2679395
6,OneRepublic,4406707
7,Bad Bunny,2679395
8,Chris Brown,4855071
9,Harry Styles,3429995


In [ ]:
orig_data.to_parquet(PROJECT_ROOT / "data" / "processed" / "orig_data_with_listeners.parquet")

data = pd.read_parquet(PROJECT_ROOT / "data" / 'processed' / 'tracks_enriched.parquet')

data['artists_listeners'] = orig_data['artists_listeners']
data.to_parquet(PROJECT_ROOT / "data" / 'processed' / 'data_with_listeners.parquet')